# IAN 620 · Module 9
## End-to-End Agentic RAG with a Downloadable Local LLM, LangChain, Chroma, DuckDuckGo, and LangGraph

This notebook builds one complete workflow from the ground up. **No paid LLM API is required for the core notebook.** A small open model is downloaded from Hugging Face and runs directly in the notebook.

**Learning path**

1. Load and inspect a small knowledge-base dataset.
2. Convert rows to LangChain documents and create chunks.
3. Create local Hugging Face embeddings.
4. Build a persistent Chroma vector store.
5. Create a retriever and test semantic search.
6. Download and run a local Qwen3 language model.
7. Build classic RAG with the local model.
8. Turn retrieval into a LangChain tool.
9. Add DuckDuckGo web search as a second tool.
10. Build a plain-Python Agentic RAG loop.
11. Build a LangGraph agent that uses the local model.
12. Optionally compare with a provider-based LangChain `create_agent` example.
13. Reload Chroma and evaluate retrieval behavior.

> Default local model: `Qwen/Qwen3-0.6B`. It downloads automatically from Hugging Face, requires no API key, and can be replaced with a larger local model later.

### Architecture used in this notebook

| Layer | What we use | Why |
|---|---|---|
| Dataset | CSV | Easy to inspect and reuse |
| Documents | `Document` | Standard LangChain data object |
| Chunking | `RecursiveCharacterTextSplitter` | Simple general-purpose chunking |
| Embeddings | `all-MiniLM-L6-v2` | Small local embedding model |
| Vector database | Chroma | Persistent local vector store |
| Retrieval | `as_retriever()` | Standard retriever interface |
| Local LLM | `Qwen/Qwen3-0.6B` | Downloadable; no API key required |
| RAG | Retriever + prompt + local LLM | Grounded generation |
| Tools | Retriever tool + DuckDuckGo | Internal knowledge + external web |
| Agent | Plain Python + LangGraph | Explicit decisions and bounded loops |
| Optional comparison | LangChain `create_agent` | High-level provider/tool-calling example |

## 1. Install the packages

Run this cell once in Colab. After installation, Colab may ask you to restart the runtime if a package was already loaded.

In [ ]:
!pip -q install -U \
    langchain \
    langgraph \
    langchain-chroma \
    langchain-community \
    langchain-huggingface \
    langchain-openai \
    chromadb \
    "sentence-transformers>=5.2.0" \
    "transformers>=4.52.4" \
    accelerate \
    duckduckgo-search

## 2. Imports and version check

Keeping the imports together makes it easier to see which library owns each part of the workflow.

In [ ]:
import os
import shutil
from pathlib import Path

import pandas as pd

import langchain
import langgraph
import chromadb

print('LangChain:', langchain.__version__)
print('LangGraph:', getattr(langgraph, '__version__', 'installed'))
print('Chroma:', chromadb.__version__)

## 3. Load the dataset

The CSV is stored in the same GitHub repository as this notebook. In Colab, we read it directly from the raw GitHub URL.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/ali2449/IAN-620/main/IAN_620_Module_9_Agentic_RAG_Knowledge_Base.csv"

kb = pd.read_csv(DATA_URL)
print('Rows:', len(kb))
print('Columns:', list(kb.columns))
kb.head()

### Inspect the knowledge base

Each row is a short policy or support document. The `source_id` gives us a stable citation identifier, and the metadata fields can later be used for filtering.

In [ ]:
kb[['source_id', 'category', 'title', 'effective_date']].head(10)

In [ ]:
kb['category'].value_counts()

## 4. Convert rows into LangChain documents

A `Document` keeps the text together with metadata. This is important because a RAG system should preserve where each piece of evidence came from.

In [ ]:
from langchain_core.documents import Document

documents = []

for _, row in kb.iterrows():
    documents.append(
        Document(
            page_content=f"{row['title']}\n{row['text']}",
            metadata={
                'source_id': row['source_id'],
                'category': row['category'],
                'title': row['title'],
                'effective_date': row['effective_date'],
                'source_url': row['source_url'],
            },
        )
    )

print('Documents:', len(documents))
print(documents[0])

## 5. Chunk the documents

The example rows are short, but we still use a text splitter so the notebook follows the same workflow used for longer documents.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
)

chunks = splitter.split_documents(documents)

print('Original documents:', len(documents))
print('Chunks:', len(chunks))
print(chunks[0].page_content)

## 6. Create local embeddings

`all-MiniLM-L6-v2` runs locally and is small enough for a teaching notebook. Embeddings turn each chunk into a numeric vector so semantically similar text can be retrieved.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={'normalize_embeddings': True},
)

sample_vector = embeddings.embed_query('return policy after delivery')
print('Embedding dimensions:', len(sample_vector))
print('First 5 values:', sample_vector[:5])

## 7. Build a persistent Chroma vector database

Chroma stores the chunk vectors and metadata. `persist_directory` lets us reload the same collection later instead of rebuilding it every time.

In [ ]:
from langchain_chroma import Chroma

CHROMA_DIR = "./ian620_module9_chroma"
COLLECTION_NAME = "ian620_module9_support"

# Start clean so re-running the notebook does not duplicate the teaching data.
shutil.rmtree(CHROMA_DIR, ignore_errors=True)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=[f'chunk-{i:03d}' for i in range(len(chunks))],
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
)

print('Chroma directory:', Path(CHROMA_DIR).resolve())
print('Stored chunks:', len(vector_store.get()['ids']))

## 8. Test semantic retrieval

Start by calling the vector store directly. This keeps retrieval separate from generation so we can inspect whether the correct evidence is being found.

In [ ]:
query = "How long do I have to return a normal item after delivery?"

results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f'--- Result {i} ---')
    print('Source:', doc.metadata['source_id'])
    print('Category:', doc.metadata['category'])
    print(doc.page_content)
    print()

## 9. Create a retriever

A retriever gives us a simple interface: give it a question and receive the most relevant documents.

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={'k': 3}
)

docs = retriever.invoke(
    "When will a refund reach the original payment method?"
)

[(d.metadata['source_id'], d.metadata['title']) for d in docs]

## 10. Download and run a local LLM — no API key

The notebook uses **Qwen3-0.6B** as the default generator. The model is downloaded automatically from Hugging Face the first time this cell runs.

- No OpenAI, Anthropic, Gemini, or other LLM API key is required.
- The standard Transformers checkpoint is about 1.5 GB.
- A GPU is recommended in Colab: **Runtime → Change runtime type → GPU**.
- CPU also works, but generation will be slower.
- For a stronger machine, you can later change the model name to `Qwen/Qwen3-1.7B`.

The model files are cached locally by Hugging Face, so later runs can reuse the downloaded model when the cache is still available.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LOCAL_MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_MODEL_NAME
)

local_model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)

print('Local model:', LOCAL_MODEL_NAME)
print('Device:', local_model.device)

In [ ]:
def local_generate(prompt, max_new_tokens=220):
    messages = [
        {
            'role': 'system',
            'content': (
                'You are a concise support assistant. '
                'Use supplied evidence carefully and do not invent facts.'
            ),
        },
        {'role': 'user', 'content': prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(local_model.device)

    with torch.no_grad():
        outputs = local_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_tokens = outputs[0][inputs['input_ids'].shape[-1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()


print(local_generate(
    'In one sentence, explain what retrieval-augmented generation is.',
    max_new_tokens=80,
))

## 11. Build classic RAG with the local model

Classic RAG follows a fixed path: **question → retrieve → context → local LLM → answer**. There is no correction loop yet, and no API key is needed.

In [ ]:
def format_context(docs):
    parts = []
    for doc in docs:
        sid = doc.metadata.get('source_id', 'unknown')
        parts.append(f'[{sid}] {doc.page_content}')
    return '\n\n'.join(parts)


def classic_rag(question):
    docs = retriever.invoke(question)
    context = format_context(docs)

    prompt = f'''
Answer the question using only the supplied context.
If the context is insufficient, say that you cannot verify the answer.
Cite source IDs such as [KB001].

CONTEXT:
{context}

QUESTION:
{question}
'''

    answer = local_generate(prompt)

    return {
        'answer': answer,
        'context': context,
        'documents': docs,
    }

In [ ]:
question = "Can I return a standard item 20 days after delivery?"
rag_result = classic_rag(question)

print(rag_result['answer'])
print('\nRetrieved sources:')
print([d.metadata['source_id'] for d in rag_result['documents']])

## 12. Turn the retriever into a LangChain tool

A retriever tool is the bridge from RAG to an agent. The model can now request retrieval as an action instead of the application always retrieving automatically.

In [ ]:
from langchain_core.tools import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    name='search_support_knowledge',
    description=(
        'Search the approved internal support knowledge base for '
        'returns, refunds, shipping, warranty, accounts, billing, '
        'subscriptions, privacy, support hours, and related policies.'
    ),
)

print('Tool name:', retriever_tool.name)
print('Tool description:', retriever_tool.description)

In [ ]:
print(
    retriever_tool.invoke(
        {"query": "What is the standard return deadline?"}
    )
)

## 13. Add DuckDuckGo web search

DuckDuckGo gives the agent an external search tool that does not require a separate API key. Keep internal knowledge and public-web evidence conceptually separate.

**Policy used in this notebook:** internal company questions should use the knowledge-base retriever. Web search is for explicitly external or current information.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

web_search = DuckDuckGoSearchRun()

try:
    web_result = web_search.invoke(
        "LangGraph StateGraph official documentation"
    )
    print(str(web_result)[:1200])
except Exception as exc:
    print('Web search could not run in this environment:')
    print(exc)

## 14. A plain-Python Agentic RAG loop

Before LangChain agents or LangGraph, make the control logic explicit. This version retrieves, grades the evidence, rewrites a weak query, and stops after a bounded number of attempts.

In [ ]:
def evidence_score(question, docs):
    if not docs:
        return 0.0

    q_words = {
        w.strip('.,?!').lower()
        for w in question.split()
        if len(w) > 3
    }
    text = ' '.join(d.page_content.lower() for d in docs)

    if not q_words:
        return 0.0

    matched = sum(word in text for word in q_words)
    return matched / len(q_words)


def rewrite_query(question):
    q = question.lower()

    if 'return' in q:
        return question + ' return window delivery deadline refund policy'
    if 'refund' in q:
        return question + ' refund processing payment method business days'
    if 'shipping' in q or 'delivery' in q:
        return question + ' shipping delivery tracking policy'

    return question + ' support policy requirements eligibility'

In [ ]:
def run_plain_agentic_rag(question, max_steps=3, threshold=0.25):
    query = question
    trace = []
    docs = []

    for step in range(1, max_steps + 1):
        docs = retriever.invoke(query)
        score = evidence_score(question, docs)

        trace.append({
            'step': step,
            'action': 'retrieve_and_grade',
            'query': query,
            'score': round(score, 3),
            'sources': [d.metadata['source_id'] for d in docs],
        })

        if score >= threshold:
            break

        new_query = rewrite_query(question)
        trace.append({
            'step': step,
            'action': 'rewrite',
            'query': new_query,
        })
        query = new_query

    context = format_context(docs)

    prompt = f'''
Use only the verified evidence below.
If the evidence is not sufficient, say so.
Cite source IDs.

EVIDENCE:
{context}

QUESTION:
{question}
'''

    answer = local_generate(prompt)

    return {
        'answer': answer,
        'documents': docs,
        'trace': trace,
    }

In [ ]:
plain_result = run_plain_agentic_rag("Can a normal item be returned after 20 days?")

print(plain_result['answer'])
pd.DataFrame(plain_result['trace'])

## 15. Optional: compare with LangChain `create_agent`

The notebook is already fully usable without an API key: classic RAG, the plain-Python agent loop, and the LangGraph workflow all use the downloaded local model.

`create_agent` is included as an **optional comparison** because high-level tool-calling agents depend on a chat-model integration that supports native tool calling. This section uses OpenAI only if you choose to provide a key. You can skip it and continue directly to LangGraph.

In [ ]:
import getpass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

provider_agent = None

if os.getenv('OPENAI_API_KEY'):
    provider_llm = ChatOpenAI(model='gpt-5-mini')
    provider_agent = create_agent(
        model=provider_llm,
        tools=[retriever_tool, web_search],
        system_prompt=(
            'Use search_support_knowledge for internal policies. '
            'Use DuckDuckGo only for explicitly public/current information. '
            'Never let web results override internal company policy.'
        ),
    )

print(
    'Optional provider agent ready:',
    provider_agent is not None,
)
print(
    'The local-model LangGraph agent below does not require an API key.'
)

In [ ]:
if provider_agent is not None:
    result = provider_agent.invoke({
        'messages': [
            {
                'role': 'user',
                'content': 'What is the standard return window in our knowledge base?',
            }
        ]
    })
    print(result['messages'][-1].content)
else:
    print('Optional provider example skipped. No API key is required for the rest of the notebook.')

In [ ]:
if provider_agent is not None:
    result = provider_agent.invoke({
        'messages': [
            {
                'role': 'user',
                'content': 'Search the public web for the current LangGraph documentation and explain StateGraph.',
            }
        ]
    })
    print(result['messages'][-1].content)
else:
    print('Continue to the local-model LangGraph agent below.')

## 16. Build a no-API-key agent with LangGraph + the local LLM

Now we build the agent explicitly with LangGraph. **This section uses the downloaded Qwen3 model for final generation, so it does not require an LLM API key.**

The graph has two evidence routes:

- **Internal:** retrieve → grade → generate or rewrite → retrieve again
- **External:** DuckDuckGo search → generate

The graph also keeps a step counter and prevents an unlimited rewrite loop.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class RAGState(TypedDict):
    question: str
    query: str
    route: str
    documents: list[dict]
    web_results: str
    score: float
    steps: int
    answer: str
    allow_web: bool

In [ ]:
def serialize_docs(docs):
    return [
        {
            'text': d.page_content,
            'source_id': d.metadata.get('source_id', 'unknown'),
            'category': d.metadata.get('category', ''),
            'source_url': d.metadata.get('source_url', ''),
        }
        for d in docs
    ]


def route_node(state: RAGState):
    q = state['question'].lower()
    web_words = {'latest', 'current', 'today', 'web', 'internet', 'public'}
    wants_web = state.get('allow_web', False) and any(
        word in q for word in web_words
    )

    return {
        'route': 'web' if wants_web else 'retrieve',
        'query': state.get('query') or state['question'],
    }


def route_after_start(state: RAGState):
    return state['route']

In [ ]:
def retrieve_node(state: RAGState):
    docs = retriever.invoke(state['query'])
    return {
        'documents': serialize_docs(docs),
        'steps': state['steps'] + 1,
    }


def grade_node(state: RAGState):
    docs = [
        Document(
            page_content=d['text'],
            metadata={
                'source_id': d['source_id'],
                'category': d['category'],
                'source_url': d['source_url'],
            },
        )
        for d in state['documents']
    ]

    score = evidence_score(state['question'], docs)
    return {'score': score}


def route_after_grade(state: RAGState):
    if state['score'] >= 0.25:
        return 'generate'

    if state['steps'] >= 2:
        return 'generate'

    return 'rewrite'

In [ ]:
def rewrite_node(state: RAGState):
    return {
        'query': rewrite_query(state['question'])
    }


def web_node(state: RAGState):
    try:
        text = web_search.invoke(state['query'])
    except Exception as exc:
        text = f'Web search unavailable: {exc}'

    return {
        'web_results': str(text),
        'steps': state['steps'] + 1,
    }


def generate_node(state: RAGState):
    internal_context = '\n\n'.join(
        f"[{d['source_id']}] {d['text']}"
        for d in state.get('documents', [])
    )

    web_context = state.get('web_results', '')

    context = internal_context
    if web_context:
        context += '\n\nPUBLIC WEB RESULTS:\n' + web_context

    if not context.strip():
        return {'answer': 'I could not find enough evidence to answer.'}

    prompt = f'''
Answer from the retrieved evidence.
Keep internal policy evidence and public web evidence clearly separated.
Do not treat public web results as internal company policy.
If evidence is insufficient, say so.

EVIDENCE:
{context}

QUESTION:
{state['question']}
'''

    return {
        'answer': local_generate(
            prompt,
            max_new_tokens=260,
        )
    }

In [ ]:
builder = StateGraph(RAGState)

builder.add_node('route', route_node)
builder.add_node('retrieve', retrieve_node)
builder.add_node('grade', grade_node)
builder.add_node('rewrite', rewrite_node)
builder.add_node('web', web_node)
builder.add_node('generate', generate_node)

builder.add_edge(START, 'route')

builder.add_conditional_edges(
    'route',
    route_after_start,
    {
        'retrieve': 'retrieve',
        'web': 'web',
    },
)

builder.add_edge('retrieve', 'grade')

builder.add_conditional_edges(
    'grade',
    route_after_grade,
    {
        'generate': 'generate',
        'rewrite': 'rewrite',
    },
)

builder.add_edge('rewrite', 'retrieve')
builder.add_edge('web', 'generate')
builder.add_edge('generate', END)

graph = builder.compile()
print('LangGraph compiled.')

### Inspect the graph structure

`draw_mermaid()` gives a text representation of the graph. You can paste the output into a Mermaid renderer, or use it as a quick way to inspect nodes and edges.

In [ ]:
print(graph.get_graph().draw_mermaid())

### Run the graph with an internal knowledge-base question

In [ ]:
internal_state = {
    'question': 'Can a standard item be returned 20 days after delivery?',
    'query': '',
    'route': '',
    'documents': [],
    'web_results': '',
    'score': 0.0,
    'steps': 0,
    'answer': '',
    'allow_web': False,
}

internal_result = graph.invoke(
    internal_state,
    {'recursion_limit': 10},
)

print(internal_result['answer'])
print('\nRoute:', internal_result['route'])
print('Steps:', internal_result['steps'])
print('Score:', round(internal_result['score'], 3))
print('Sources:', [d['source_id'] for d in internal_result['documents']])

### Run the graph with an explicitly external/current question

This example allows the web route. The graph should not use public web search unless `allow_web=True` and the wording signals that current/external information is wanted.

In [ ]:
web_state = {
    'question': 'Search the public web for the current LangGraph documentation and explain StateGraph.',
    'query': '',
    'route': '',
    'documents': [],
    'web_results': '',
    'score': 0.0,
    'steps': 0,
    'answer': '',
    'allow_web': True,
}

web_graph_result = graph.invoke(
    web_state,
    {'recursion_limit': 10},
)

print(web_graph_result['answer'])
print('\nRoute:', web_graph_result['route'])
print('Steps:', web_graph_result['steps'])

## 17. Optional local deployment: GGUF / llama.cpp / Ollama

The Transformers model above is easiest in Colab. For a laptop or desktop, a **GGUF** file is often convenient because it can run with llama.cpp-compatible applications.

Official lightweight option:

- Repository: `Qwen/Qwen3-0.6B-GGUF`
- `Qwen3-0.6B-Q8_0.gguf` is about 639 MB.
- It can be run through llama.cpp, and the official model page also provides an Ollama command.

Example terminal commands:

```bash
# llama.cpp / llama CLI
llama cli -hf Qwen/Qwen3-0.6B-GGUF:Q8_0

# Ollama
ollama run hf.co/Qwen/Qwen3-0.6B-GGUF:Q8_0
```

This is optional; the notebook itself already runs the Transformers version directly.

## 18. Reload the persisted Chroma collection

Persistence matters because a real application should not rebuild embeddings every time it starts.

In [ ]:
reloaded_store = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
)

print('Reloaded chunks:', len(reloaded_store.get()['ids']))

reloaded_docs = reloaded_store.similarity_search(
    'How long does the limited warranty last?',
    k=3,
)

[(d.metadata['source_id'], d.metadata['title']) for d in reloaded_docs]

## 19. Evaluate retrieval with a small test set

Before evaluating final answers, test whether retrieval finds the expected evidence. This makes it easier to identify whether a failure comes from retrieval or generation.

In [ ]:
tests = [
    ('What is the normal return deadline?', 'returns'),
    ('How long does refund processing take?', 'refunds'),
    ('When should shipping damage be reported?', 'damaged_items'),
    ('How long is the limited warranty?', 'warranty'),
    ('How do I get a password reset link?', 'accounts'),
    ('When can I cancel a subscription?', 'subscriptions'),
]

rows = []

for question, expected_category in tests:
    docs = retriever.invoke(question)
    categories = [d.metadata['category'] for d in docs]
    sources = [d.metadata['source_id'] for d in docs]

    rows.append({
        'question': question,
        'expected_category': expected_category,
        'retrieved_categories': ', '.join(categories),
        'sources': ', '.join(sources),
        'hit': expected_category in categories,
    })

evaluation = pd.DataFrame(rows)
evaluation

In [ ]:
print(
    'Retrieval hit rate:',
    f"{evaluation['hit'].mean():.0%}"
)

## 20. Troubleshooting checklist

### Retrieval returns the wrong material
- Inspect the raw chunks first.
- Try a more specific query.
- Adjust `k`.
- Add metadata filters.
- Consider a stronger embedding model or reranker.

### The agent loops
- Log the query, retrieved sources, grade, selected route, and step count.
- Make sure the rewrite actually changes something.
- Make sure the grader can realistically pass.
- Keep an application step limit and a LangGraph `recursion_limit`.

### The agent uses web search for internal policy
- Strengthen the system prompt.
- Route deterministically before the model when the distinction is obvious.
- Do not allow public web evidence to override approved internal policy.

### Chroma duplicates documents after re-running
- Use stable IDs.
- Reuse the existing collection intentionally, or delete the teaching directory before rebuilding.

### DuckDuckGo fails
- Internet access may be unavailable in the runtime.
- Treat search as an external dependency and handle failure explicitly.

## 21. What you built

You now have the complete no-API-key path:

**CSV dataset → LangChain Documents → chunks → local embeddings → Chroma → retriever → downloadable Qwen3 local LLM → classic RAG → retriever tool → DuckDuckGo → plain-Python agent loop → LangGraph agent → persistence → evaluation**

The optional provider `create_agent` section is included only for comparison. The main RAG and LangGraph workflow runs with the downloaded local model.

The most important architectural lesson is that LangChain and LangGraph do not replace the RAG components. They organize the same retriever, tools, prompts, model calls, state, decisions, and stop conditions in more reusable ways.